# **Initialization**

In [1]:
"""Start"""

'Start'

In [2]:
# ==========================================
# 0. IMPORTS & SETUP
# ==========================================

# --- 1. Standard Library Imports ---
import ast
import contextlib
import gc
import multiprocessing
import os
import re
import shutil
import sys
import time
import traceback
from functools import lru_cache

# --- 2. Data Science & Math Imports ---
import numpy as np
import pandas as pd
from numpy.linalg import eigh
from scipy.optimize import linear_sum_assignment
from scipy.sparse.csgraph import minimum_spanning_tree

# --- 3. Optimization & Solvers Imports ---
import modified_didppy as m_dp
import pulp
from ortools.linear_solver import pywraplp

# --- 4. Custom Library Path Setup ---
# Adjust this path if moving the script to a different location
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\Evolutionary_algorithm"

if os.path.exists(LIBRARY_PARENT_PATH) and LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)

# --- 5. Custom Library Imports ---
try:
    from evolutionary_algorithm_lib import (
        compile_chromosome_to_useable_function,
        combining_modified_didppy_solver_with_chromosome
    )
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    
    print("✅ Libraries imported successfully.")

except ImportError as e:
    print(f"❌ Library Import Error: {e}")
    print(f"   Ensure '{LIBRARY_PARENT_PATH}' is correct and contains the library.")

✅ Libraries imported successfully.


# **Configuration & Data input**

In [3]:
# --- GLOBAL VARIABLES (Initialize with Dummy Data) ---
# We create these so the functions in Cell 3 don't crash if checked early.
# These will be overwritten by the loop in Cell 4.
current_num_locations = 5
current_travel_cost = [[0.0]*5 for _ in range(5)]

print("✅ Globals initialized.")

# --- BATCH UTILITIES ---
def get_processed_instances(csv_path, logs_dir):
    """
    Returns a set of instances that exist in BOTH the CSV summary and the logs folder.
    """
    # 1. Get instances from CSV
    csv_instances = set()
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if 'Instance' in df.columns:
                csv_instances = set(df['Instance'].unique())
        except:
            pass # CSV read failed, assume empty

    # 2. Get instances from Log Files
    log_instances = set()
    if os.path.exists(logs_dir):
        for filename in os.listdir(logs_dir):
            if filename.endswith("_log.txt"):
                # Extract "0.txt" from "0.txt_log.txt"
                instance_name = filename.replace("_log.txt", "")
                log_instances.add(instance_name)

    # 3. Return Intersection (Must be in BOTH to be considered "Done")
    return csv_instances.intersection(log_instances)

def append_result_to_csv(result_dict, csv_path):
    df = pd.DataFrame([result_dict])
    df.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)

print("✅ Configuration set.")

def read_tsp_cappart_format(file_path):
    with open(file_path, 'r') as f:
        values = f.read().split()
    iterator = iter(values)
    n = int(next(iterator))
    c = []
    for i in range(n):
        row = []
        for j in range(n):
            row.append(int(float(next(iterator))))
        c.append(row)
    return n, c

# Cell 3.5: Data Cleanup Utility

def clean_batch_data(csv_path, logs_dir):
    """
    Ensures consistency between the CSV summary and the Log files.
    1. Removes duplicate instances in CSV (keeps last).
    2. Removes CSV rows if the corresponding Log file is missing.
    3. Deletes Log files if the corresponding CSV row is missing.
    """
    print("🧹 Starting Data Cleanup...")
    
    # 1. Load CSV
    if not os.path.exists(csv_path):
        print("   -> CSV not found. Nothing to clean in CSV.")
        # If CSV missing but logs exist, we might want to clear logs, 
        # but usually better to leave them or delete manually to be safe.
        return 

    try:
        df = pd.read_csv(csv_path)
    except pd.errors.EmptyDataError:
        print("   -> CSV is empty.")
        return

    original_count = len(df)
    
    # 2. Deduplicate CSV (Keep the last run)
    df.drop_duplicates(subset=['Instance'], keep='last', inplace=True)
    dedup_count = len(df)
    if original_count > dedup_count:
        print(f"   -> Removed {original_count - dedup_count} duplicate rows from CSV.")

    # 3. Remove CSV rows without matching Log files
    valid_indices = []
    instances_in_csv = set()
    
    for index, row in df.iterrows():
        instance_name = row['Instance']
        expected_log = os.path.join(logs_dir, f"{instance_name}_log.txt")
        
        if os.path.exists(expected_log):
            valid_indices.append(index)
            instances_in_csv.add(instance_name)
        else:
            print(f"   -> Removing CSV row for '{instance_name}' (Log file missing).")
            
    # Filter dataframe to keep only valid rows
    df_clean = df.loc[valid_indices]
    
    # Save cleaned CSV
    df_clean.to_csv(csv_path, index=False)
    print(f"   -> CSV saved. Current number of row is: {len(df_clean)} (was {original_count}).")

    # 4. Remove Orphan Log files (Log exists, but not in CSV)
    if os.path.exists(logs_dir):
        files = os.listdir(logs_dir)
        for filename in files:
            if filename.endswith("_log.txt"):
                instance_from_log = filename.replace("_log.txt", "")
                
                if instance_from_log not in instances_in_csv:
                    file_path = os.path.join(logs_dir, filename)
                    try:
                        os.remove(file_path)
                        print(f"   -> Deleted orphan log: {filename} (Not in CSV).")
                    except OSError as e:
                        print(f"   -> Error deleting {filename}: {e}")

    print("✨ Data Cleanup Complete.\n")


✅ Globals initialized.
✅ Configuration set.


# **Model and dual bounds declaration**

In [4]:
def creation_of_didp_model_function():
    """
    Creates the CVRP DIDP model and returns it along with necessary metadata 
    for the heuristic functions.
    """
    n = current_num_locations
    c = current_travel_cost
    
    # 2. Initialize Model
    # Note: Ensure float_cost matches your data. Your snippet used False (Int), 
    # so we explicitly cast distances to Int in the reader.
    model = m_dp.Model(maximize=False, float_cost=True)

    customer = model.add_object_type(number=n)

    # 3. State Variables
    # U: Unvisited set (excluding depot 0)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
    # i: Current location
    location = model.add_element_var(object_type=customer, target=0)

    # 4. Resource Tables
    travel_time = model.add_float_table(c)

    # 5. Transitions
    # Visit customer j
    for j in range(1, n):
        visit = m_dp.Transition(
            name="visit {}".format(j),
            cost=travel_time[location, j] + m_dp.FloatExpr.state_cost(),
            preconditions=[unvisited.contains(j)],
            effects=[
                (unvisited, unvisited.remove(j)),
                (location, j),
            ],
        )
        model.add_transition(visit)

    # Return to depot
    # Note: Removed 'time' effect from your snippet as it wasn't defined in the variables
    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time[location, 0] + m_dp.FloatExpr.state_cost(),
        effects=[
            (location, 0),
        ],
        preconditions=[unvisited.is_empty(), location != 0],
    )
    model.add_transition(return_to_depot)

    # 6. Base Case
    model.add_base_case([unvisited.is_empty(), location == 0])

    # 8. Create Bundle (Model + Metadata)
    # This metadata dict allows your heuristics (like MST or assignment) 
    # to access the raw matrix data later.
    metadata = {
        "num_nodes": n,
        "distance_matrix": c,
        "unvisited_var": unvisited,
        "location_var": location,
        # Add other keys if your dual bounds need them
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

In [5]:
def create_persistent_lp_relaxation_3_index_dual_bounds(metadata):
    """
    1. Persistent 3-Index TSP Relaxation (MTZ Formulation).
    - Variables: x[i,j] (Flow), u[i] (Position/Potential).
    - Logic: Enforces connectivity via MTZ constraints.
    - Pattern: Cached internal worker '_solve_3idx'.
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    # --- INITIALIZATION ---
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver: return lambda state: 0.0
    infinity = solver.infinity()

    # Variables
    x = {}
    u = {}
    for i in range(n_nodes):
        u[i] = solver.NumVar(0, n_nodes, f'u_{i}')
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    # Constraints (Mutable)
    cons_out = {}
    cons_in = {}
    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'out_{i}')
        c_in = solver.Constraint(0, 0, f'in_{i}')
        for j in range(n_nodes):
            if i != j:
                c_out.SetCoefficient(x[(i, j)], 1)
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_out[i] = c_out
        cons_in[i] = c_in

    # MTZ Constraints (Static)
    # u_i - u_j + N*x_ij <= N - 1
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                c = solver.Constraint(-infinity, n_nodes - 1, f'mtz_{i}_{j}')
                c.SetCoefficient(u[i], 1)
                c.SetCoefficient(u[j], -1)
                c.SetCoefficient(x[(i, j)], n_nodes)

    # Objective
    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # --- CACHED WORKER ---
    @lru_cache(maxsize=10000)
    def _solve_3idx(active_tuple):
        # Tuple structure: (current_node, sorted_unvisited...)
        current_node = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_node)
        active_set.add(0) # Depot

        max_u = len(active_set) # Max position in path

        for i in range(n_nodes):
            if i in active_set:
                if i == current_node:
                    # Start: Out=1, In=0, u=0
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(0, 0)
                    u[i].SetBounds(0, 0)
                elif i == 0:
                    # End: Out=0, In=1, u=max
                    cons_out[i].SetBounds(0, 0)
                    cons_in[i].SetBounds(1, 1)
                    u[i].SetBounds(1, max_u)
                else:
                    # Mid: Out=1, In=1, u in [1, max]
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(1, 1)
                    u[i].SetBounds(1, max_u)
            else:
                # Inactive
                cons_out[i].SetBounds(0, 0)
                cons_in[i].SetBounds(0, 0)
                u[i].SetBounds(0, 0)

        solver.SetTimeLimit(100)
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # --- WRAPPER ---
    def h_lp_relaxation_3_idx(state):
        unvisited = state[unvisited_var]
        curr = state[location_var]
        if not unvisited and curr == 0: return 0.0
        
        # Key must distinguish current node (start of path)
        key = (curr,) + tuple(sorted(list(unvisited)))
        return _solve_3idx(key)

    return h_lp_relaxation_3_idx


def create_persistent_lp_relaxation_2_index_dual_bounds(metadata):
    """
    2. Persistent 2-Index TSP Relaxation (Assignment/Flow Formulation).
    - Variables: x[i,j] (Flow). NO MTZ variables.
    - Logic: Relaxed connectivity (Subtours allowed). Faster than 3-Index.
    - Pattern: Cached internal worker '_solve_2idx'.
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    # --- INITIALIZATION ---
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver: return lambda state: 0.0

    # Variables
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    # Constraints (Degree/Flow)
    cons_out = {}
    cons_in = {}
    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'out_{i}')
        c_in = solver.Constraint(0, 0, f'in_{i}')
        for j in range(n_nodes):
            if i != j:
                c_out.SetCoefficient(x[(i, j)], 1)
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_out[i] = c_out
        cons_in[i] = c_in

    # Objective
    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # --- CACHED WORKER ---
    @lru_cache(maxsize=10000)
    def _solve_2idx(active_tuple):
        current_node = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_node)
        active_set.add(0)

        for i in range(n_nodes):
            if i in active_set:
                if i == current_node:
                    # Start: Out=1, In=0
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(0, 0)
                elif i == 0:
                    # End: Out=0, In=1
                    cons_out[i].SetBounds(0, 0)
                    cons_in[i].SetBounds(1, 1)
                else:
                    # Mid: Out=1, In=1
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(1, 1)
            else:
                # Inactive
                cons_out[i].SetBounds(0, 0)
                cons_in[i].SetBounds(0, 0)

        solver.SetTimeLimit(100)
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # --- WRAPPER ---
    def h_lp_relaxation_2_idx(state):
        unvisited = state[unvisited_var]
        curr = state[location_var]
        if not unvisited and curr == 0: return 0.0
        
        key = (curr,) + tuple(sorted(list(unvisited)))
        return _solve_2idx(key)

    return h_lp_relaxation_2_idx

In [6]:
def dual_bound_expression_function(didp_bundle):
    """ 
    Registry containing ALL heuristics (Combinatorial + LP) for TSP.
    """
    model, metadata = didp_bundle
    
    # Extract metadata
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list)
    num_nodes = metadata['num_nodes']

    # Pre-computation
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    min_outgoing_arr = np.min(masked_cost, axis=1)
    min_incoming_arr = np.min(masked_cost, axis=0)

    # --- Initialize LP Bounds ---
    h_lp_relaxation_3_idx = create_persistent_lp_relaxation_3_index_dual_bounds(metadata)
    h_lp_relaxation_2_idx = create_persistent_lp_relaxation_2_index_dual_bounds(metadata)

    # ==========================================
    # COMBINATORIAL BOUNDS (Internal Workers)
    # ==========================================

    # --- Degree Average Bound ---
    @lru_cache(maxsize=100000)
    def _calc_degree(active_tuple):
        nodes = list(active_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)
        mins_in = np.min(sub_mat, axis=0) 
        mins_out = np.min(sub_mat, axis=1)
        # Current (0) excludes Incoming, Depot (-1) excludes Outgoing
        sum_in = np.sum(mins_in[1:])
        sum_out = np.sum(mins_out[:-1])
        return float(0.5 * (sum_in + sum_out))

    def h_degree_average(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        
        # Tuple: (current, U..., depot)
        active_list = [curr] + sorted(list(U))
        if 0 not in active_list: active_list.append(0)
        return _calc_degree(tuple(active_list))

    # --- Global Min Flow ---
    @lru_cache(maxsize=100000)
    def _calc_min_flow_static(unvisited_tuple):
        val_out = sum(min_outgoing_arr[u] for u in unvisited_tuple)
        val_in = sum(min_incoming_arr[u] for u in unvisited_tuple)
        return val_out, val_in

    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        
        val_out, val_in = _calc_min_flow_static(tuple(sorted(list(U))))
        if curr != 0:
            val_out += min_outgoing_arr[curr]
            val_in += min_incoming_arr[0]
        return float(max(val_out, val_in))

    # --- MST Bound ---
    @lru_cache(maxsize=100000)
    def _calc_mst(unvisited_tuple):
        if not unvisited_tuple: return 0.0
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    def h_mst(state):
        U = state[unvisited_var]
        return _calc_mst(tuple(sorted(list(U))))

    # --- 1-Tree Bound ---
    @lru_cache(maxsize=100000)
    def _calc_1tree(unvisited_tuple):
        subset = list(unvisited_tuple)
        depot_edges = sorted(cost_matrix[0, subset])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        if len(subset) > 1:
            sub_mat = cost_matrix[np.ix_(subset, subset)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0
        return float(mst_val + e1 + e2)

    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_1tree(tuple(sorted(list(U))))

    # --- Assignment Bound ---
    @lru_cache(maxsize=100000)
    def _calc_assignment(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        np.fill_diagonal(assign_mat, np.inf)
        row, col = linear_sum_assignment(assign_mat)
        return float(assign_mat[row, col].sum())

    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_assignment(tuple(sorted(list(U))))

    # --- Eigenvalue Bound ---
    @lru_cache(maxsize=100000)
    def _calc_eigen(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        N = len(nodes)
        if N < 2: return 0.0
        
        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1))
        P = np.eye(N) - (one @ one.T) / N
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
        
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])
        
        phi = 0.0
        if N > 1:
            if N % 2 == 1:
                num_terms = (N - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                     phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
            else:
                num_sum_terms = N // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N-2 < len(eigvals): phi += 2 * eigvals[N - 2]
                elif N > 1 and N-2 < len(eigvals):
                    phi = 2 * eigvals[N - 2]
        return float(phi)

    def h_eigen(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_eigen(tuple(sorted(list(U))))

    # Return valid registry
    return automatic_creation_of_dual_bounds_registry(locals())

# Execution Line
dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())
display(dual_bound_functions_registry)

{'h_lp_relaxation_3_idx': <function __main__.create_persistent_lp_relaxation_3_index_dual_bounds.<locals>.h_lp_relaxation_3_idx(state)>,
 'h_lp_relaxation_2_idx': <function __main__.create_persistent_lp_relaxation_2_index_dual_bounds.<locals>.h_lp_relaxation_2_idx(state)>,
 'h_degree_average': <function __main__.dual_bound_expression_function.<locals>.h_degree_average(state)>,
 'h_global_min_flow': <function __main__.dual_bound_expression_function.<locals>.h_global_min_flow(state)>,
 'h_mst': <function __main__.dual_bound_expression_function.<locals>.h_mst(state)>,
 'h_1tree': <function __main__.dual_bound_expression_function.<locals>.h_1tree(state)>,
 'h_assignment': <function __main__.dual_bound_expression_function.<locals>.h_assignment(state)>,
 'h_eigen': <function __main__.dual_bound_expression_function.<locals>.h_eigen(state)>}

# **Execution**

In [7]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
SOLVER_TIME_LIMIT = 2
ENABLE_BATCH_MODE = True
SINGLE_TARGET_INSTANCE = "98.txt"
n_50_signal = False

# Path Setup
if n_50_signal:
    DATA_DIR = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Datasets\n50"
    EA_INPUT_CSV_PATH = "result_of_ea_TSP_dual_bounds_50_cus.csv"
    VERIFICATION_OUTPUT_CSV = "TSP_EA_dual_bound_verification_results_50_cus.csv"
    LOG_DIR = "solver_logs_TSP_EA_dual_bounds_50_cus" # <--- Log Folder
else:
    DATA_DIR = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Datasets\n20"
    EA_INPUT_CSV_PATH = "result_of_ea_TSP_dual_bounds_20_cus.csv"
    VERIFICATION_OUTPUT_CSV = "TSP_EA_dual_bound_verification_results_20_cus.csv"
    LOG_DIR = "solver_logs_TSP_EA_dual_bounds_20_cus" # <--- Log Folder

INPUT_CSV_PATH = EA_INPUT_CSV_PATH

# Create Log Directory
os.makedirs(LOG_DIR, exist_ok=True)

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
class SolverLogger:
    """
    Context manager to redirect low-level stdout (FD 1) to a file.
    This captures C++/Rust output (like CABS logs) that Python's 'sys.stdout' misses.
    """
    def __init__(self, filename):
        self.filename = filename
        self.log_file = None
        self.original_stdout_fd = None

    def __enter__(self):
        # Open the log file
        self.log_file = open(self.filename, 'w', encoding='utf-8', buffering=1) # Line buffered
        try:
            # 1. Get the file descriptor for the real stdout (usually 1)
            self.stdout_fd = sys.stdout.fileno()
            # 2. Save a copy of the original stdout to restore later
            self.original_stdout_fd = os.dup(self.stdout_fd)
            # 3. Redirect stdout to our file
            os.dup2(self.log_file.fileno(), self.stdout_fd)
        except Exception as e:
            print(f"⚠️ Logger Warning: Could not redirect C-level output ({e}). Logs might be incomplete.")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # Restore stdout
        if self.original_stdout_fd is not None:
            os.dup2(self.original_stdout_fd, self.stdout_fd)
            os.close(self.original_stdout_fd)
        
        if self.log_file:
            self.log_file.close()

def save_result_immediately(output_data, csv_path):
    """
    Saves a single row to CSV immediately and forces disk write.
    """
    try:
        df_new = pd.DataFrame([output_data])
        
        if os.path.exists(csv_path):
            try:
                df_existing = pd.read_csv(csv_path)
                # Deduplicate
                if output_data['Instance'] in df_existing['Instance'].values:
                    df_existing = df_existing[df_existing['Instance'] != output_data['Instance']]
                # Concat
                df_final = pd.concat([df_existing, df_new], ignore_index=True)
                df_final.to_csv(csv_path, index=False)
            except Exception:
                df_new.to_csv(csv_path, mode='a', header=False, index=False)
        else:
            df_new.to_csv(csv_path, index=False)
            
        print(f"   ✅ Saved {output_data['Instance']} to disk.")
        
    except Exception as e:
        print(f"   ⚠️ Save Failed: {e}")

# ==========================================
# 3. MAIN EXECUTION
# ==========================================
def run_verification():
    print("🔹 Initializing Robust Verification Script (Single Process)...")

    # --- Load Targets ---
    if not os.path.exists(INPUT_CSV_PATH):
        print(f"❌ Input CSV not found: {INPUT_CSV_PATH}")
        return
    
    df_input = pd.read_csv(INPUT_CSV_PATH)
    targets = []
    
    if ENABLE_BATCH_MODE:
        processed = set()
        if os.path.exists(VERIFICATION_OUTPUT_CSV):
            try: processed = set(pd.read_csv(VERIFICATION_OUTPUT_CSV)['Instance'].values)
            except: pass
            
        for _, row in df_input.iterrows():
            if row['Instance'] not in processed:
                targets.append(row)
        print(f"📝 Processing {len(targets)} remaining instances.")
    else:
        target_row = df_input[df_input['Instance'] == SINGLE_TARGET_INSTANCE]
        if not target_row.empty: targets.append(target_row.iloc[0])

    # --- Processing Loop ---
    for i, row in enumerate(targets):
        instance_name = row['Instance']
        safe_name = instance_name.replace(".txt", "")
        log_file_path = os.path.join(LOG_DIR, f"solver_log_{safe_name}.txt")

        print(f"\n[{i+1}/{len(targets)}] Processing {instance_name}...")
        print(f"   📂 Logs will be saved to: {log_file_path}")

        gc.collect()

        try:
            # Parse Chromosome
            chromosome_list = ast.literal_eval(row['Best_Chromosome'])
            
            # Check Data File
            file_path = os.path.join(DATA_DIR, instance_name)
            if not os.path.exists(file_path):
                print("   ❌ Data file missing.")
                continue

            # Update Globals
            global current_num_locations, current_travel_cost
            current_num_locations, current_travel_cost = read_tsp_cappart_format(file_path)

            # --- RUN SOLVER WITH LOGGING ---
            # The 'with SolverLogger' block redirects all print outputs (including Rust) to the file
            with SolverLogger(log_file_path):
                
                # 1. Create Registry (Lightweight)
                temp_model, temp_meta = creation_of_didp_model_function()
                
                # Print Code (Useful for debugging, goes to log file now)
                if i == 0:
                    print("--- Chromosome Logic ---")
                    compile_chromosome_to_useable_function(
                        {'chromosome': chromosome_list, 'fitness': 0}, 
                        dual_bound_functions_dict=dual_bound_expression_function((temp_model, temp_meta)),
                        print_code=True
                    )
                    print("------------------------")
                
                del temp_model, temp_meta
                gc.collect()

                # 2. Execute Solver
                result_tuple = combining_modified_didppy_solver_with_chromosome(
                    chromosome_list,
                    creation_of_didp_model_function,
                    dual_bound_expression_function,
                    solver_time_limit=SOLVER_TIME_LIMIT,
                    output_other_result=True,
                    print_timing_stats=True,
                    solver_quite=False # <--- Output is enabled, Logger captures it to file
                )

            # --- END LOGGING BLOCK ---
            # Output is back to console now

            # Unpack
            if result_tuple is None:
                cost, is_opt, gen, exp, stats = float('inf'), False, 0, 0, {}
            else:
                cost, is_opt, gen, exp, stats = result_tuple

            output_data = {
                "Instance": instance_name,
                "Objective Value": cost,
                "Nodes Expanded": exp,
                "Nodes Generated": gen,
                "Optimality": is_opt,
                "Infeasibility": (cost == float('inf')),
                "Total Times (s)": stats.get('total_duration', 0.0) if stats else 0,
                "Bridging Time (s)": stats.get('total_bridge_time', 0.0) if stats else 0,
                "Python Bound Calculation Time (s)": stats.get('python_dual_bound_calc_time', 0.0) if stats else 0
            }

            print(f"   -> Result: Cost={cost}, Time={output_data['Total Times (s)']:.2f}s")
            save_result_immediately(output_data, VERIFICATION_OUTPUT_CSV)

        except MemoryError:
            print(f"❌ 🧠 OOM (Out of Memory) on {instance_name}!")
            err_data = {
                "Instance": instance_name, "Objective Value": "OOM Error",
                "Nodes Expanded": -1, "Nodes Generated": -1,
                "Optimality": "Error", "Infeasibility": "Error",
                "Total Times (s)": 0, "Bridging Time (s)": 0, "Python Bound Calculation Time (s)": 0
            }
            save_result_immediately(err_data, VERIFICATION_OUTPUT_CSV)
            gc.collect()

        except Exception as e:
            print(f"❌ Error: {e}")
        
        finally:
            if 'result_tuple' in locals(): del result_tuple
            gc.collect()

if __name__ == "__main__":
    run_verification()

🔹 Initializing Robust Verification Script (Single Process)...
❌ Input CSV not found: result_of_ea_TSP_dual_bounds_20_cus.csv
